# J02 — L'eau du sol : teneur en eau et état énergétique

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 2 : unités du potentiel, tensiomètres et charge hydraulique, loi de Jurin, étalonnage TDR et stock d'eau*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Les données sont dans le dossier `data/`. Convention du cours : $z$ positif vers le haut, origine à la surface du sol ;
charge hydraulique $H = h + z$ (cm) ; eau à 20 °C ($\rho_w = 998{,}2$ kg/m³, $g = 9{,}81$ m/s², $\sigma = 72{,}8$ mN/m).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})
RHO_W, G, SIGMA = 998.2, 9.81, 0.0728      # kg/m³, m/s², N/m (20 °C)

## Exercice 1 — Unités du potentiel et échelle pF  *(≈ 10 min)*

Le potentiel de l'eau s'exprime par unité de masse ($\psi$, J/kg), de volume ($\Psi = \rho_w \psi$, Pa) ou de poids
($h = \psi/g$, m ou cm de colonne d'eau). Rappels : $1$ kPa $= 1000/(\rho_w g)$ m $\approx 10{,}2$ cm ; $1$ bar $= 100$ kPa ;
$\mathrm{pF} = \log_{10}(|h|$ en cm$)$.

1. Écrire `convertir(valeur, de, vers)` qui convertit une valeur de potentiel entre les unités `"kPa"`, `"bar"`, `"cm"`, `"m"` et
   `"J/kg"` (passer par une unité pivot, par exemple le Pa).
2. Écrire `pF(h_cm)` (renvoyer `nan` si $h \ge 0$).
3. Construire la table des états remarquables pour $\Psi = -1, -10, -33, -100, -1500$ kPa : valeur en bar, cm, m, J/kg et pF.
4. Vérifier que pF $= \log_{10}(|\Psi|_{\mathrm{kPa}}) + 1{,}01$.

In [ ]:
def convertir(valeur, de, vers):
    """Convertit un potentiel entre kPa, bar, cm, m, J/kg (pivot : Pa)."""
    en_Pa = {"kPa": 1e3, "bar": 1e5, "m": RHO_W * G, "cm": RHO_W * G / 100, "J/kg": RHO_W}
    # À COMPLÉTER : valeur * en_Pa[de] / en_Pa[vers]
    pass

def pF(h_cm):
    # À COMPLÉTER
    pass

# 3. table des états remarquables
etats = {"saturation proche": -1, "capacité au champ (sable)": -10, "capacité au champ": -33,
         "pF 3 (approx.)": -100, "point de flétrissement": -1500}
# À COMPLÉTER : DataFrame avec une ligne par état

## Exercice 2 — Tensiomètres : charge hydraulique et sens de l'écoulement  *(≈ 20 min)*

`data/J02_tensiometres.csv` : six tensiomètres à jauge à vide installés à 15, 30, 45, 60, 90 et 120 cm de profondeur dans un loam.
Pour chaque tensiomètre : `lecture_kPa` (succion $S$ lue par la jauge, positive) et `L_col_cm` (hauteur de la jauge au-dessus
de la bougie). Deux dates : après une pluie, puis après sept jours sans pluie.

1. Calculer la charge de pression à la bougie $h = -10{,}2\,S + L$ (cm), la cote $z = -$profondeur et la charge hydraulique $H = h + z$.
2. Tracer $h(z)$ et $H(z)$ pour les deux dates (profondeur en ordonnée).
3. Entre tensiomètres voisins, calculer le gradient $\Delta H/\Delta z = (H_{\text{haut}} - H_{\text{bas}})/(z_{\text{haut}} - z_{\text{bas}})$
   et en déduire le sens du flux (gradient $> 0$ : vers le bas ; $< 0$ : vers le haut) et son intensité relative.
4. Localiser, à la seconde date, le plan de flux nul (changement de signe du gradient : interpoler la profondeur où $\Delta H/\Delta z = 0$).

In [ ]:
tens = pd.read_csv("data/J02_tensiometres.csv")
KPA_CM = 1e3 / (RHO_W * G) * 100      # cm de colonne d'eau par kPa

# 1. h, z, H
tens["h_cm"] = # À COMPLÉTER
tens["z_cm"] = # À COMPLÉTER
tens["H_cm"] = # À COMPLÉTER

# 2. profils
fig, ax = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for date, g in tens.groupby("date"):
    ax[0].plot(g["h_cm"], g["z_cm"], "o-", label=date)
    ax[1].plot(g["H_cm"], g["z_cm"], "o-", label=date)
# À COMPLÉTER : axes, légendes

# 3. gradients entre tensiomètres voisins
def gradients(g):
    g = g.sort_values("profondeur_cm")
    # À COMPLÉTER : DataFrame (intervalle, dH/dz, sens)
    pass

# 4. plan de flux nul

## Exercice 3 — Loi de Jurin et distribution des tailles de pores  *(≈ 15 min)*

La loi de Jurin relie la charge de pression $h$ au rayon du plus gros pore encore plein d'eau :
$|h| = 2\sigma\cos\gamma/(\rho_w g\, r)$, soit $r\,[\mathrm{cm}] \approx 0{,}149/|h|\,[\mathrm{cm}]$ à 20 °C ($\gamma = 0$).

1. Écrire `rayon(h_cm, T=20)` (rayon équivalent en µm) et `h_de_rayon(r_um, T=20)` (cm), avec $\sigma(T)$ interpolée
   dans la table : 0 °C : 75,6 ; 10 °C : 74,2 ; 20 °C : 72,8 ; 30 °C : 71,2 ; 40 °C : 69,6 mN/m. Vérifier le facteur 0,149 à 20 °C
   et calculer la remontée capillaire pour $r$ = 1, 10, 100 et 1000 µm.
2. `data/J02_retention.csv` : courbes $\theta(h)$ (drainage) d'un sable et d'un loam. Calculer $r$ pour chaque point, puis la
   densité de distribution des tailles de pores $f(r) = \mathrm{d}\theta/\mathrm{d}\log_{10} r$ (`np.gradient`). Tracer $f$ en fonction de $r$ (échelle log).
3. Pour chaque sol : rayon modal, fraction du volume poral ($\theta_s - \theta_r$, avec $\theta_r = \theta(-10^5$ cm$)$)
   correspondant aux macropores ($r > 30$ µm) et aux pores $< 0{,}1$ µm.

In [ ]:
T_tab = np.array([0, 10, 20, 30, 40]); sig_tab = np.array([75.6, 74.2, 72.8, 71.2, 69.6]) * 1e-3   # N/m

def rayon(h_cm, T=20):
    """Rayon de pore équivalent (µm) pour une charge de pression h (cm)."""
    # À COMPLÉTER
    pass

def h_de_rayon(r_um, T=20):
    # À COMPLÉTER
    pass

ret = pd.read_csv("data/J02_retention.csv")
# 2. rayon équivalent et densité dθ/dlog10 r
# 3. rayon modal, fractions macropores / pores < 0,1 µm

## Exercice 4 — Étalonnage TDR et stock d'eau d'un profil  *(≈ 15 min)*

1. `data/J02_tdr_etalonnage.csv` : 30 couples ($\varepsilon_a$, $\theta_{\text{grav}}$) mesurés sur un loam riche en matière
   organique. Ajuster un polynôme de degré 3 $\theta = a_0 + a_1\varepsilon_a + a_2\varepsilon_a^2 + a_3\varepsilon_a^3$
   (`np.polyfit`) et le comparer à l'équation de Topp et al. (1980) :
   $\theta = -5{,}3\times10^{-2} + 2{,}92\times10^{-2}\varepsilon_a - 5{,}5\times10^{-4}\varepsilon_a^2 + 4{,}3\times10^{-6}\varepsilon_a^3$.
   Calculer le RMSE et le biais moyen de chaque étalonnage par rapport aux mesures gravimétriques.
2. `data/J02_tdr_profils.csv` : $\varepsilon_a$ mesurée à 8 profondeurs (10 à 100 cm) à trois dates ; 32 mm de pluie sont tombés
   entre les dates 2 et 3. Convertir en $\theta$ avec les deux étalonnages, tracer les profils.
3. Calculer le stock d'eau $W$ (mm) sur 0–100 cm par la méthode des trapèzes (`np.trapezoid`), en supposant $\theta$ constant entre
   la surface et la première sonde. Comparer les deux étalonnages et commenter la variation de stock entre les dates.

In [ ]:
def topp(eps):
    return -5.3e-2 + 2.92e-2 * eps - 5.5e-4 * eps**2 + 4.3e-6 * eps**3

cal = pd.read_csv("data/J02_tdr_etalonnage.csv")
# 1. polynôme de degré 3 et comparaison
coef = # À COMPLÉTER : np.polyfit(...)
poly_local = np.poly1d(coef)
# RMSE, biais

# 2. profils
prof = pd.read_csv("data/J02_tdr_profils.csv")
dates = [c for c in prof.columns if c != "profondeur_cm"]

# 3. stock d'eau 0-100 cm (trapèzes) ; ajouter le point z = 0 avec theta = theta(10 cm)
def stock_mm(z_cm, theta):
    # À COMPLÉTER
    pass

## Exercice 5 — Bonus — potentiel osmotique  *(≈ facultatif)*

Calculer le potentiel osmotique de solutions de NaCl de 1 à 500 mmol/L par la loi de van 't Hoff
$\Psi_o = -i\,c\,R\,T$ ($i = 2$, $c$ en mol/m³, $R = 8{,}314$ J/(mol K), $T = 293$ K), l'exprimer en kPa et en m d'eau,
puis comparer à la règle empirique $\Psi_o$ [kPa] $\approx -36\,\mathrm{CE}$ [dS/m] avec $\mathrm{CE} \approx 0{,}107\,c^{0{,}98}$ dS/m ($c$ en mmol/L).

In [ ]:
c_mmol = np.array([1, 5, 10, 50, 100, 500])
# À COMPLÉTER : van 't Hoff, conversion, règle -36 CE

## Pour aller plus loin

* Exercice 2 : estimer le flux réel entre tensiomètres en supposant $K(h)$ de Mualem–van Genuchten pour le loam (Jour 5).
* Exercice 3 : refaire le calcul avec un angle de contact de 60° (sol hydrophobe) ; que devient la distribution ?
* Exercice 4 : propager une incertitude de ±0,5 sur $\varepsilon_a$ (Monte-Carlo) jusqu'au stock d'eau.